In [1]:
import os
import json
import shutil
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)
from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/MKB_10_NOOW.csv',
)

dataset.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [7]:
MAIN_MODALITY = '@text'

In [8]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [9]:
dataset.get_dictionary()

artm.Dictionary(name=182e58a5-5844-4544-9010-55753ab1d3c8, num_entries=244551)

In [10]:
dictionary = dataset.get_dictionary()

In [11]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=182e58a5-5844-4544-9010-55753ab1d3c8, num_entries=244551)


In [12]:
print(dictionary)

artm.Dictionary(name=182e58a5-5844-4544-9010-55753ab1d3c8, num_entries=46873)


In [13]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=182e58a5-5844-4544-9010-55753ab1d3c8, num_entries=22608)

In [14]:
dataset._cached_dict = dictionary

In [15]:
dataset.get_dictionary()

artm.Dictionary(name=182e58a5-5844-4544-9010-55753ab1d3c8, num_entries=22608)

In [16]:
def is_good(coherence):
    return 1.5041606723306686 <= coherence

def is_bad(coherence):
    return coherence <= 1.2499085596596853

In [10]:
dataset.get_dictionary()

artm.Dictionary(name=f05c88de-f9ad-4732-913b-7f53c294e642, num_entries=244551)

In [11]:
dictionary = dataset.get_dictionary()

In [12]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=f05c88de-f9ad-4732-913b-7f53c294e642, num_entries=244551)


In [13]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=f05c88de-f9ad-4732-913b-7f53c294e642, num_entries=22608)

In [14]:
dataset._cached_dict = dictionary

In [15]:
dataset.get_dictionary()

artm.Dictionary(name=f05c88de-f9ad-4732-913b-7f53c294e642, num_entries=22608)

In [17]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [18]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 4.41 s, sys: 155 ms, total: 4.57 s
Wall time: 4.52 s


In [19]:
co_occurences.shape

(22608, 22608)

In [20]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [21]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [22]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [23]:
NUM_TOPICS = 50  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [24]:
NUM_GOOD_TOPICS_THRESHOLD = NUM_TOPICS - 5

In [25]:
NUM_GOOD_TOPICS_THRESHOLD

45

In [44]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}



    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PWT,  # TODO: changed from PTW to check if good topics are fixed
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        # TODO: "harsh" regularization may result in the fact that some topics have None intratext coherence
        # assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        # assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        for x, y in zip(_values, current_intra_topic_coherences.values()):
            if x is None:
                assert y is None
            elif y is None:
                assert x is None
            else:
                assert abs(x - y) <= 1e-6

        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        if all(v is not None for v in current_intra_topic_coherences.values()):
            value = float(np.median(list(current_intra_topic_coherences.values())))
        else:
            value = None

        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    
    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [27]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [28]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [29]:
TOPIC_INDICES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [33]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [34]:
! ls results/mkb10/

ablation_study		iterative_100000.json	    lda.json
decorrelation.json	iterative2_100000000	    plsa.json
iterative_100000	iterative2_1000000000	    sparse.json
iterative_1000000	iterative2_1000000000.json  tless.json
iterative_1000000.json	iterative2_100000000.json


In [44]:
! tail -n 50 results/mkb10/iterative2_100000000.json

            "17": 0.8008765599168247,
            "18": 0.8327142599268654,
            "19": 0.5969504754349373
        },
        "num_topics": {
            "good": 17,
            "bad": 0,
            "not_good": 3,
            "total_bad": 10
        }
    },
    {
        "scores": {
            "perplexity": 2766.8095703125,
            "coherence_20": 0.8920629718870099,
            "diversity_euclidean": 0.08743469937513604,
            "diversity_jensenshannon": 0.7110107864693572,
            "diversity_hellinger": 0.8375688488788172,
            "diversity_cosine": 0.86820693821513
        },
        "topic_coherences": {
            "0": 0.762131239154706,
            "1": 0.9428906252054199,
            "2": 0.7755873398182704,
            "3": 0.8066311856951885,
            "4": 0.9669159891260591,
            "5": 0.8937573669068252,
            "6": 0.9410456312491235,
            "7": 0.8048923241212161,
            "8": 0.7625888750824479,
            "9": 0.741937

In [30]:
# BEST_TAUS = [100000, 100000000]
BEST_TAUS =   [100000, 100000000]

# Also nice
# 1000000 1000000000

In [46]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [31]:
SAVE_FOLDER = 'results50_intra/mkb10'

os.makedirs(SAVE_FOLDER, exist_ok=True)

In [48]:
SAVE_FOLDER

'results50/mkb10'

In [49]:
BEST_TAUS

[100000, 100000000]

In [36]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer

DECORRELATION_TAUS = [BEST_TAUS[0]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if (new_result['scores']['diversity_jensenshannon'] == -1
                    or new_result['scores']['toplen_ptw'] is None):  # TODO: say about it
                print('Early stopping because of corrupted topics')

                break

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]



            assert not any(t in new_bad_topic_names for t in good_topic_names)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))
                
                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]

            

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.035681259057055235
sparse_theta_sp: -0.396209187014688
decorrelation: 0.01
None
num_topics: {'good': 10, 'good_fair': 10, 'bad': 4, 'not_good': 40, 'total_bad': 4}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f646bd579d0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f634e1e1550>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f63558780d0>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.044601573821319046
sparse_theta_sp: -0.49526148376835993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_1', 'topic_7', 'topic_11', 'topic_16', 'topic_25', 'topic_27', 'topic_40', 'topic_43', 'topic_45', 'topic_48'] are not in new good topics ['topic_0', 'topic_1', 'topic_10', 'topic_11', 'topic_16', 'topic_27', 'topic_36', 'topic_40', 'topic_48']. Manually marking them as good.
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 13, 'good_fair

...

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f64be7874c0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f646b7e48b0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f63484ac370>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.06371653403045578
sparse_theta_sp: -0.7075164053833713
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_1', 'topic_5', 'topic_7', 'topic_10', 'topic_11', 'topic_16', 'topic_22', 'topic_23', 'topic_25', 'topic_26', 'topic_27', 'topic_33', 'topic_34', 'topic_36', 'topic_38', 'topic_40', 'topic_43', 'topic_45', 'topic_47', 'topic_48', 'topic_49'] are not in new good topics ['topic_0', 'topic_1', 'topic_5', 'topic_7', 'topic_10', 'topic_11', 'topic_12', 'topic_1

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f6488acd9a0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f646b6cf7f0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f6488acdc10>}


AssertionError: 

In [ ]:
1

In [37]:
results.keys()

dict_keys([100000])

In [38]:
len(results[100000])

14

In [43]:
results[100000][0]

{'scores': {'perplexity': 1887.1734619140625,
  'coherence_20': 0.7833287611226992,
  'toplen_ptw': 1.3937002943672534,
  'diversity_euclidean': 0.07592577696337832,
  'diversity_jensenshannon': 0.6654680724493455,
  'diversity_hellinger': 0.7785246561326203,
  'diversity_cosine': 0.8114537834294857},
 'topic_coherences': {0: 0.620169332330262,
  1: 1.113181347952901,
  2: 0.5454635690775651,
  3: 0.804055033069975,
  4: 0.44260089820813725,
  5: 0.41467012258747915,
  6: 0.8547757649205125,
  7: 1.072577222382621,
  8: 0.5510948223208315,
  9: 0.5653871930972489,
  10: 0.8099938274684965,
  11: 0.5313094624011118,
  12: 0.8144043399078691,
  13: 0.7918027308018832,
  14: 0.645021918060935,
  15: 0.7307178513836462,
  16: 1.199071381252546,
  17: 0.7789857597110078,
  18: 0.5372719492468814,
  19: 1.0081683322566697,
  20: 1.0382961530833528,
  21: 0.7818309066356389,
  22: 0.7748263526848076,
  23: 1.581344537500257,
  24: 0.6143802496738281,
  25: 0.7791184792768634,
  26: 0.68048592

In [39]:
SAVE_FOLDER

'results50_intra/mkb10'

In [40]:
print('Saving results')

for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

Saving results


In [41]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  lda_with_cohs.json     tless_with_cohs.json
iterative_100000	      plsa_with_cohs.json
iterative_100000.json	      sparse_with_cohs.json


In [45]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer2

DECORRELATION_TAUS = [BEST_TAUS[1]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative2_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if (new_result['scores']['diversity_jensenshannon'] == -1
                    or new_result['scores']['toplen_ptw'] is None):  # TODO: say about it
                print('Early stopping because of corrupted topics')

                break

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]



            assert not any(t in new_bad_topic_names for t in good_topic_names)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))
                
                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]

            

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.035681259057055235
sparse_theta_sp: -0.396209187014688
decorrelation: 0.01
None
num_topics: {'good': 10, 'good_fair': 10, 'bad': 4, 'not_good': 40, 'total_bad': 4}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f634ab54040>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f634ab54400>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f6355823d60>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.044601573821319046
sparse_theta_sp: -0.49526148376835993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_1', 'topic_7', 'topic_11', 'topic_16', 'topic_25', 'topic_27', 'topic_40', 'topic_43', 'topic_45', 'topic_48'] are not in new good topics ['topic_0', 'topic_1', 'topic_11', 'topic_16', 'topic_25', 'topic_27', 'topic_36', 'topic_40', 'topic_45', 'topic_48']. Manually marking them as good.
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'g

...

test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f62e9c09df0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f6328faba00>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f644d5d0130>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.05575196727664881
sparse_theta_sp: -0.61907685471045
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_1', 'topic_3', 'topic_7', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', 'topic_37', 'topic_40', 'topic_43', 'topic_45', 'topic_48', 'topic_49'] are not in new good topics ['topic_0', 'topic_1', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', 'topic_37', 

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f65603b8d90>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f646b6cdfd0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f6469c2f1c0>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.05575196727664881
sparse_theta_sp: -0.61907685471045
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_1', 'topic_3', 'topic_7', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', 'topic_37', 'topic_40', 'topic_43', 'topic_45', 'topic_48', 'topic_49'] are not in new good topics ['topic_0', 'topic_1', 'topic_3', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', '

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f6488acd7c0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f64ec9a0100>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f646bbcedf0>}
test_8
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.05575196727664881
sparse_theta_sp: -0.61907685471045
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_1', 'topic_3', 'topic_7', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', 'topic_37', 'topic_40', 'topic_43', 'topic_45', 'topic_48', 'topic_49'] are not in new good topics ['topic_0', 'topic_1', 'topic_3', 'topic_11', 'topic_16', 'topic_23', 'topic_25', 'topic_27', 'topic_32', 'topic_33', 'topic_36', '

In [46]:
results.keys()

dict_keys([100000000])

In [47]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  iterative2_100000000	 plsa_with_cohs.json
iterative_100000	      iterative2_100000000.json  sparse_with_cohs.json
iterative_100000.json	      lda_with_cohs.json	 tless_with_cohs.json


In [57]:
! ls $SAVE_FOLDER

decorrelation.json     iterative2_100000000	  plsa.json
iterative_100000       iterative2_100000000.json  sparse.json
iterative_100000.json  lda.json			  tless.json


## Ablation Study

In [58]:
DECORRELATION_TAU = BEST_TAUS[0]

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [59]:
SAVE_FOLDER + f'/ablation_study'

'results50/mkb10/ablation_study'

In [60]:
os.makedirs(SAVE_FOLDER + f'/ablation_study', exist_ok=True)

In [61]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.035681259057055235
sparse_theta_sp: -0.396209187014688
decorrelation: 0.01
None
num_topics: {'good': 11, 'bad': 6, 'not_good': 39, 'total_bad': 6}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f66175dcfa0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f66141d3e20>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.045745203919301584
sparse_theta_sp: -0.5079604961726769
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 24, 'bad': 3, 'not_good': 26, 'total_bad': 9}
Removing: results50/mkb10/ablation_study/iterative_100000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f655e6f7e50>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 31, 'bad': 5, 'not_good': 19, 'total_bad': 122}
Removing: results50/mkb10/ablation_study/iterative_100000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f655e64f0d0>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 31, 'bad': 8, 'not_good': 19, 'total_bad': 130}
Removing: results50/mkb10/ablation_study/iterative_100000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f66441f8190>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 32, 'bad': 4, 'not_good': 18, 'total_bad': 134}
Removing: results50/mkb10/ablation_study/iterative_100000_1-0-0/18
Saving results


In [62]:
1

1

In [63]:
DECORRELATION_TAU = BEST_TAUS[1]

In [64]:
DECORRELATION_TAU

100000000

In [65]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.035681259057055235
sparse_theta_sp: -0.396209187014688
decorrelation: 0.01
None
num_topics: {'good': 11, 'bad': 6, 'not_good': 39, 'total_bad': 6}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6619c2f430>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f66141458e0>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.045745203919301584
sparse_theta_sp: -0.5079604961726769
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 15, 'bad': 3, 'not_good': 35, 'total_bad': 9}
Removing: results50/mkb10/ablation_study/iterative2_100000000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6617156580>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 31, 'bad': 5, 'not_good': 19, 'total_bad': 122}
Removing: results50/mkb10/ablation_study/iterative2_100000000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f661416fd60>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 31, 'bad': 8, 'not_good': 19, 'total_bad': 130}
Removing: results50/mkb10/ablation_study/iterative2_100000000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f66175cbb50>}
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.09389805015014535
sparse_theta_sp: -1.0426557553018103
decorrelation: 0.01
fix: 1000000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 32, 'bad': 4, 'not_good': 18, 'total_bad': 134}
Removing: results50/mkb10/ablation_study/iterative2_100000000_1-0-0/18
Saving results


In [66]:
1

1

In [68]:
results.keys()

dict_keys([(1, 0, 1), (1, 1, 0), (1, 0, 0)])